# MLflow Trace Debug Notebook

Use this notebook to inspect one Databricks MLflow evaluation run from Python instead of the UI.

Default flow:
1. Parse the run URL and connect to MLflow.
2. Inspect run metadata and the logged evaluation artifacts.
3. Search the run's traces with optional MLflow trace filters.
4. Pick one trace and flatten its spans and assessments for debugging.

Useful trace filter examples:
- `trace.status = 'ERROR'`
- `trace.execution_time_ms > 5000`
- `span.name ILIKE '%tool%'`
- `trace.name ILIKE '%hadm_%'`

In [ ]:
from __future__ import annotations

import json
import re
from urllib.parse import parse_qs, urlparse

import mlflow
import pandas as pd
from mlflow import MlflowClient

pd.set_option("display.max_colwidth", 220)


def parse_run_url(run_url: str) -> dict[str, str | None]:
    text = str(run_url or "").strip()
    parsed = urlparse(text)
    match = re.search(r"/ml/experiments/(?P<experiment_id>[^/]+)/runs/(?P<run_id>[^/]+)", parsed.path)
    if match is None:
        raise ValueError(f"Could not parse experiment/run IDs from URL: {run_url}")
    query = parse_qs(parsed.query)
    return {
        "run_url": text,
        "tracking_host": f"{parsed.scheme}://{parsed.netloc}",
        "workspace_id": query.get("o", [None])[0],
        "experiment_id": match.group("experiment_id"),
        "run_id": match.group("run_id"),
    }


def safe_to_dict(value):
    if value is None:
        return None
    if isinstance(value, dict):
        return value
    if isinstance(value, list):
        return [safe_to_dict(item) for item in value]
    to_dict = getattr(value, "to_dict", None)
    if callable(to_dict):
        return to_dict()
    to_dictionary = getattr(value, "to_dictionary", None)
    if callable(to_dictionary):
        return to_dictionary()
    data = getattr(value, "__dict__", None)
    if isinstance(data, dict):
        return dict(data)
    return value


def try_parse_json(value):
    if value is None:
        return None
    if isinstance(value, (dict, list, int, float, bool)):
        return value
    text = str(value).strip()
    if not text:
        return None
    try:
        return json.loads(text)
    except Exception:
        return value


def short_text(value, max_chars: int = 240):
    normalized = try_parse_json(value)
    if normalized is None:
        return None
    if isinstance(normalized, (dict, list)):
        text = json.dumps(normalized, ensure_ascii=True, sort_keys=True)
    else:
        text = str(normalized)
    compact = " ".join(text.split())
    return compact if len(compact) <= max_chars else f"{compact[:max_chars]}..."


def list_artifacts_recursive(
    client: MlflowClient,
    run_id: str,
    path: str = "",
    max_depth: int = 4,
    depth: int = 0,
) -> list[dict[str, object]]:
    rows: list[dict[str, object]] = []
    for artifact in client.list_artifacts(run_id, path):
        rows.append(
            {
                "depth": depth,
                "path": artifact.path,
                "is_dir": artifact.is_dir,
                "file_size": getattr(artifact, "file_size", None),
            }
        )
        if artifact.is_dir and depth + 1 < max_depth:
            rows.extend(
                list_artifacts_recursive(
                    client=client,
                    run_id=run_id,
                    path=artifact.path,
                    max_depth=max_depth,
                    depth=depth + 1,
                )
            )
    return rows


def load_run_json(run_id: str, artifact_path: str):
    try:
        return mlflow.artifacts.load_dict(f"runs:/{run_id}/{artifact_path}"), None
    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"


def get_root_span_name(trace) -> str | None:
    spans = list(getattr(getattr(trace, "data", None), "spans", []) or [])
    if not spans:
        return None
    span_rows = [safe_to_dict(span) or {} for span in spans]
    for row in span_rows:
        if not row.get("parent_span_id"):
            return row.get("name")
    return span_rows[0].get("name")


def summarize_trace(trace) -> dict[str, object]:
    info = safe_to_dict(getattr(trace, "info", None)) or {}
    assessments = info.get("assessments") or []
    spans = list(getattr(getattr(trace, "data", None), "spans", []) or [])
    return {
        "trace_id": info.get("trace_id"),
        "state": info.get("state"),
        "request_time_ms": info.get("request_time"),
        "execution_duration_ms": info.get("execution_duration"),
        "client_request_id": info.get("client_request_id"),
        "root_span_name": get_root_span_name(trace),
        "span_count": len(spans),
        "assessment_count": len(assessments),
        "request_preview": short_text(info.get("request_preview"), max_chars=160),
        "response_preview": short_text(info.get("response_preview"), max_chars=160),
    }


def extract_span_payload(span, attribute_key: str, property_name: str):
    direct = getattr(span, property_name, None)
    if direct not in (None, ""):
        return direct
    span_dict = safe_to_dict(span) or {}
    attributes = span_dict.get("attributes") or {}
    return try_parse_json(attributes.get(attribute_key))


def flatten_spans(trace, preview_chars: int = 300) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    for span in list(getattr(getattr(trace, "data", None), "spans", []) or []):
        span_dict = safe_to_dict(span) or {}
        attributes = span_dict.get("attributes") or {}
        status = span_dict.get("status")
        status_code = status.get("status_code") if isinstance(status, dict) else status
        rows.append(
            {
                "trace_id": span_dict.get("trace_id"),
                "span_id": span_dict.get("span_id"),
                "parent_span_id": span_dict.get("parent_span_id"),
                "name": span_dict.get("name"),
                "span_type": attributes.get("mlflow.spanType"),
                "status": status_code,
                "start_time_unix_nano": span_dict.get("start_time_unix_nano"),
                "end_time_unix_nano": span_dict.get("end_time_unix_nano"),
                "input_preview": short_text(
                    extract_span_payload(span, "mlflow.spanInputs", "inputs"),
                    max_chars=preview_chars,
                ),
                "output_preview": short_text(
                    extract_span_payload(span, "mlflow.spanOutputs", "outputs"),
                    max_chars=preview_chars,
                ),
                "attribute_keys": sorted(attributes.keys()),
            }
        )
    frame = pd.DataFrame(rows)
    if not frame.empty:
        frame = frame.sort_values(
            ["start_time_unix_nano", "name", "span_id"],
            kind="stable",
        ).reset_index(drop=True)
    return frame


def flatten_assessments(trace) -> pd.DataFrame:
    info = safe_to_dict(getattr(trace, "info", None)) or {}
    assessments = info.get("assessments") or []
    rows: list[dict[str, object]] = []
    for assessment in assessments:
        source = safe_to_dict(getattr(assessment, "source", None))
        assessment_dict = safe_to_dict(assessment) or {}
        rows.append(
            {
                "assessment_id": assessment_dict.get("assessment_id"),
                "name": assessment_dict.get("name"),
                "value": safe_to_dict(getattr(assessment, "value", None)),
                "source_type": None if source is None else source.get("source_type"),
                "source_id": None if source is None else source.get("source_id"),
                "span_id": assessment_dict.get("span_id"),
                "run_id": assessment_dict.get("run_id"),
                "rationale": short_text(assessment_dict.get("rationale"), max_chars=200),
                "metadata": safe_to_dict(assessment_dict.get("metadata")),
                "valid": assessment_dict.get("valid"),
                "error": safe_to_dict(assessment_dict.get("error")),
            }
        )
    return pd.DataFrame(rows)

In [ ]:
RUN_URL = "https://adb-2498687920111167.7.azuredatabricks.net/ml/experiments/2534872744749110/runs/bdffd941a5b54f22a05bf69b5f15e69e/evaluations?o=2498687920111167"
TRACKING_URI_OVERRIDE = None
TRACE_SEARCH_FILTER = None
TRACE_MAX_RESULTS = 200
TRACE_ARTIFACT_ROOT = "evaluation"
SELECT_TRACE_ID = None
SPAN_PREVIEW_CHARS = 700
DISABLE_NOTEBOOK_TRACE_UI = True

target = parse_run_url(RUN_URL)
if TRACKING_URI_OVERRIDE:
    mlflow.set_tracking_uri(TRACKING_URI_OVERRIDE)
if DISABLE_NOTEBOOK_TRACE_UI and hasattr(mlflow, "tracing") and hasattr(mlflow.tracing, "disable_notebook_display"):
    mlflow.tracing.disable_notebook_display()

target

In [ ]:
client = MlflowClient()
EXPERIMENT_ID = target["experiment_id"]
RUN_ID = target["run_id"]

experiment = client.get_experiment(EXPERIMENT_ID)
mlflow.set_experiment(experiment_id=EXPERIMENT_ID)
run = mlflow.get_run(RUN_ID)

{
    "tracking_uri": mlflow.get_tracking_uri(),
    "tracking_host_from_url": target["tracking_host"],
    "workspace_id_from_url": target["workspace_id"],
    "experiment_id": experiment.experiment_id,
    "experiment_name": experiment.name,
    "run_id": run.info.run_id,
    "run_name": run.data.tags.get("mlflow.runName"),
    "status": run.info.status,
    "artifact_uri": run.info.artifact_uri,
    "start_time": run.info.start_time,
    "end_time": run.info.end_time,
}

In [ ]:
selected_tag_keys = [
    key
    for key in run.data.tags
    if not key.startswith("mlflow.") or key in {"mlflow.runName", "mlflow.note.content"}
]

{
    "params": run.data.params,
    "metrics": run.data.metrics,
    "selected_tags": {key: run.data.tags[key] for key in selected_tag_keys},
}

In [ ]:
artifact_tree_df = pd.DataFrame(
    list_artifacts_recursive(
        client=client,
        run_id=RUN_ID,
        path=TRACE_ARTIFACT_ROOT,
        max_depth=4,
    )
)
artifact_tree_df

In [ ]:
evaluation_summary, evaluation_summary_error = load_run_json(
    RUN_ID,
    f"{TRACE_ARTIFACT_ROOT}/icd_react_v2_summary.json",
)
evaluation_rows_payload, evaluation_rows_error = load_run_json(
    RUN_ID,
    f"{TRACE_ARTIFACT_ROOT}/icd_react_v2_rows.json",
)
genai_eval_payload, genai_eval_error = load_run_json(
    RUN_ID,
    f"{TRACE_ARTIFACT_ROOT}/icd_react_v2_genai_eval.json",
)

evaluation_rows_df = pd.DataFrame((evaluation_rows_payload or {}).get("rows", []))
genai_eval_df = pd.DataFrame((genai_eval_payload or {}).get("rows", []))

{
    "summary_load_error": evaluation_summary_error,
    "rows_load_error": evaluation_rows_error,
    "genai_eval_load_error": genai_eval_error,
    "evaluation_summary": evaluation_summary,
    "evaluation_rows_shape": evaluation_rows_df.shape,
    "genai_eval_shape": genai_eval_df.shape,
}

In [ ]:
row_preview_columns = [
    column_name
    for column_name in [
        "hadm_id",
        "subject_id",
        "note_id",
        "status",
        "precision",
        "recall",
        "expected_icd_codes",
        "predicted_icd_codes",
        "error",
    ]
    if column_name in evaluation_rows_df.columns
]

if evaluation_rows_df.empty:
    evaluation_rows_df
else:
    evaluation_rows_df[row_preview_columns].sort_values(
        [column_name for column_name in ["status", "hadm_id", "note_id"] if column_name in row_preview_columns],
        kind="stable",
    ).reset_index(drop=True)

In [ ]:
trace_list = mlflow.search_traces(
    run_id=RUN_ID,
    locations=[EXPERIMENT_ID],
    filter_string=TRACE_SEARCH_FILTER,
    max_results=TRACE_MAX_RESULTS,
    return_type="list",
    include_spans=True,
)
trace_summary_df = pd.DataFrame([summarize_trace(trace) for trace in trace_list])
if not trace_summary_df.empty:
    trace_summary_df = trace_summary_df.sort_values(
        ["request_time_ms", "trace_id"],
        ascending=[False, True],
        kind="stable",
    ).reset_index(drop=True)
trace_summary_df

In [ ]:
ACTIVE_TRACE_ID = SELECT_TRACE_ID or (
    None if trace_summary_df.empty else str(trace_summary_df.iloc[0]["trace_id"])
)
if ACTIVE_TRACE_ID is None:
    raise ValueError("No trace matched the current run/filter selection.")

active_trace = mlflow.get_trace(ACTIVE_TRACE_ID, flush=True)
active_trace_info = safe_to_dict(active_trace.info) or {}

{
    "active_trace_id": ACTIVE_TRACE_ID,
    "root_span_name": get_root_span_name(active_trace),
    "state": active_trace_info.get("state"),
    "execution_duration_ms": active_trace_info.get("execution_duration"),
    "assessment_count": len(active_trace_info.get("assessments") or []),
    "trace_metadata": active_trace_info.get("trace_metadata"),
    "tags": active_trace_info.get("tags"),
}

In [ ]:
span_df = flatten_spans(active_trace, preview_chars=SPAN_PREVIEW_CHARS)
span_df

In [ ]:
assessment_df = flatten_assessments(active_trace)
assessment_df

In [ ]:
{
    "trace_info": safe_to_dict(active_trace.info),
    "first_span": None if span_df.empty else safe_to_dict(active_trace.data.spans[0]),
    "first_assessment": None if assessment_df.empty else safe_to_dict((active_trace.info.assessments or [])[0]),
}